In [47]:
import re
import numpy as np
import pandas as pd

In [48]:
df = pd.read_csv("../data/processed/extracted/2017.csv")

In [49]:
from pathlib import Path

matches = list(Path("..").rglob("2017.csv"))  # or Path.home() / "Downloads"
print(matches)
df = pd.read_csv(matches[0])

[WindowsPath('../data/processed/extracted/2017.csv')]


In [50]:
# 1. Load 2017 dataset and strip HTML tags / non-breaking spaces from headers
df.columns = [re.sub(r"<[^>]+>", "", c).replace("\xa0", " ").strip() for c in df.columns]

In [51]:
#STEP 2: Select and rename matching 2017 columns to match the 2014 schema
# ==============================================================================
col_map = {
    "Submit Date (UTC)": "Timestamp",
    "What is your age?": "Age",
    "What is your gender?": "Gender",
    "What country do you work in?": "Country",
    "What US state or territory do you work in?": "state",
    "Are you self-employed?": "self_employed",
    "Do you have a family history of mental illness?": "family_history",
    "Have you ever sought treatment for a mental health disorder from a mental health professional?": "treatment",
    "If you have a mental health disorder, how often do you feel that it interferes with your work when NOT being treated effectively (i.e., when you are experiencing symptoms)?": "work_interfere",
    "How many employees does your company or organization have?": "no_employees",
    "Is your employer primarily a tech company/organization?": "tech_company",
    "Does your employer provide mental health benefits as part of healthcare coverage?": "benefits",
    "Do you know the options for mental health care available under your employer-provided health coverage?": "care_options",
    "Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?": "wellness_program",
    "Does your employer offer resources to learn more about mental health disorders and options for seeking help?": "seek_help",
    "Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?": "anonymity",
    "If a mental health issue prompted you to request a medical leave from work, how easy or difficult would it be to ask for that leave?": "leave",
    "Would you feel comfortable discussing a mental health issue with your coworkers?": "coworkers",
    "Would you feel comfortable discussing a mental health issue with your direct supervisor(s)?": "supervisor",
    "Would you bring up your mental health with a potential employer in an interview?": "mental_health_interview",
    "Would you be willing to bring up a physical health issue with a potential employer in an interview?": "phys_health_interview",
    "Have you observed or experienced an unsupportive or badly handled response to a mental health issue in your current or previous workplace?": "obs_consequence",
}

clean = df[list(col_map.keys())].rename(columns=col_map).copy()

In [52]:
# Derive mental_vs_physical by comparing 2017's 0-10 importance ratings (keeping NaNs)
phys_imp = df["Overall, how much importance does your employer place on physical health?"]
ment_imp = df["Overall, how much importance does your employer place on mental health?"]
clean["mental_vs_physical"] = pd.Series(
    np.where(ment_imp >= phys_imp, "Yes", "No"), index=df.index
).where(phys_imp.notna() & ment_imp.notna(), np.nan)

In [53]:
# Add placeholder NaN columns for questions not asked in the 2017 survey
for missing_col in ["remote_work", "mental_health_consequence", "phys_health_consequence"]:
    clean[missing_col] = np.nan

In [54]:
# STEP 3: Filter invalid ages (keep 14–100) and parse timestamps
# ==============================================================================
clean = clean[clean["Age"].between(14, 100)].copy()
clean["Age"] = clean["Age"].astype(int)
clean["Timestamp"] = pd.to_datetime(clean["Timestamp"])

In [55]:
# STEP 4: Group free-text Gender entries into male, female, other, or NaN
# ==============================================================================
male = {
    "male", "m", "man", "cis-male", "cis male", "male (cis)", "male, cis",
    "cis hetero male", "mail", "dude",
    "male (hey this is the tech industry you're talking about)"
}
female = {
    "female", "f", "woman", "female (cisgender)", "female (cis)",
    "f, cisgender", "cis female", "cis-female", "my sex is female.",
    "femalw", "femail"
}
uninformative = {"\\-", "contextual", "sometimes", "god king of the valajar"}

def group_gender(g):
    if pd.isna(g):
        return np.nan
    g = str(g).strip().lower()
    if g in male:          return "male"
    if g in female:        return "female"
    if g in uninformative: return np.nan
    return "other"

clean["Gender"] = clean["Gender"].map(group_gender)

In [56]:
# STEP 5: Standardize Country names and convert US state names to 2-letter codes
# ==============================================================================
clean["Country"] = clean["Country"].replace({"United States of America": "United States"})

us_state_to_abbrev = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "District of Columbia": "DC", "Florida": "FL", "Georgia": "GA", "Hawaii": "HI",
    "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY"
}
clean["state"] = clean["state"].replace(us_state_to_abbrev)

In [57]:
# STEP 6: Standardize Yes/No/Unsure/ordinal labels and set company size order
# ==============================================================================
yn_map = {0: "No", 1: "Yes", 0.0: "No", 1.0: "Yes"}
for col in ["self_employed", "treatment", "tech_company"]:
    clean[col] = clean[col].replace(yn_map)

clean = clean.replace({
    "I don't know": "Unsure",
    "Not eligible for coverage / NA": "Unsure",
    "Neither easy nor difficult": "Unsure",
    "Maybe/Not sure": "Unsure",
    "I've always been self-employed": "Unsure",
    "Difficult": "Very difficult",
    "Not applicable to me": "Not applicable",
    "Yes, I observed": "Yes",
    "Yes, I experienced": "Yes",
})

for col in ["coworkers", "supervisor"]:
    clean[col] = clean[col].replace({"Maybe": "Some of them"})

for col in ["mental_health_interview", "phys_health_interview"]:
    clean[col] = clean[col].replace({"Maybe": "Unsure"})

ORDER_EMP = ["1-5", "6-25", "26-100", "100-500", "500-1000", "More than 1000"]

emp_code_map = {
    "1-5": 1,
    "6-25": 2,
    "26-100": 3,
    "100-500": 4,
    "500-1000": 5,
    "More than 1000": 6,
}

# Make no_employees an ordered categorical column matching the 2014 dataset
clean["no_employees"] = pd.Categorical(
    clean["no_employees"], categories=ORDER_EMP, ordered=True
)

# Add numeric indicator column (1 to 6, keeping <NA> where missing)
clean["no_employees_code"] = clean["no_employees"].map(emp_code_map).astype("Int64")

# Verify value counts in 2017 compared against 2014
print("2017 no_employees counts:")
print(clean["no_employees"].value_counts(dropna=False, sort=False))
print("\n2017 no_employees_code counts:")
print(clean["no_employees_code"].value_counts(dropna=False).sort_index())

2017 no_employees counts:
no_employees
1-5                20
6-25               86
26-100            128
100-500           203
500-1000           48
More than 1000    158
NaN               111
Name: count, dtype: int64

2017 no_employees_code counts:
no_employees_code
1        20
2        86
3       128
4       203
5        48
6       158
<NA>    111
Name: count, dtype: Int64


In [58]:
# STEP 7: Match exact 2014 column order, add year, and save clean_2017.csv
# ==============================================================================
clean["year"] = 2017
target_cols = [
    "Timestamp", "Age", "Gender", "Country", "state", "self_employed",
    "family_history", "treatment", "work_interfere", "no_employees",
    "remote_work", "tech_company", "benefits", "care_options",
    "wellness_program", "seek_help", "anonymity", "leave",
    "mental_health_consequence", "phys_health_consequence", "coworkers",
    "supervisor", "mental_health_interview", "phys_health_interview",
    "mental_vs_physical", "obs_consequence", "year"
]
output_path = "../data/processed/clean_2017.csv"
clean_2017 = clean[target_cols].reset_index(drop=True)
clean_2017.to_csv(output_path, index=False)

print("Saved:", output_path, "| Shape:", clean_2017.shape)

Saved: ../data/processed/clean_2017.csv | Shape: (754, 27)


In [59]:
print(df.head(5))

                                  #  Are you self-employed?  \
0  e49fe87572831232dcfa51b376b22039                       0   
1  a1eede444ac024928d247a8372d54931                       0   
2  37d3fd67f62bd1e0a2dea4f9cd440d98                       0   
3  519b759442c1cab0e9b5a8a1acb1b216                       0   
4  ef0af4927b575b1a3e607c11ca37870e                       1   

  How many employees does your company or organization have?  \
0                                            100-500           
1                                            100-500           
2                                               6-25           
3                                     More than 1000           
4                                                NaN           

   Is your employer primarily a tech company/organization?  \
0                                                1.0         
1                                                1.0         
2                                                1